In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from typing import TypedDict, Literal

In [ ]:
load_dotenv()

In [ ]:
model = ChatGroq(model="openai/gpt-oss-20b")

In [ ]:
class SentimentSchema(BaseModel):

    sentiment: Literal['positive', 'negative'] = Field(description="Sentiment of the review") 

In [ ]:
model_with_structured_output = model.with_structured_output(SentimentSchema)

In [ ]:
response = model_with_structured_output.invoke("What is the sentiment of the following review -> The software is too bad ")
response

In [ ]:
class ReviewState(TypedDict):
    review: str
    sentiment: Literal['positive', 'negative']
    diagnosis: dict # contains issue_type, tone , urgency
    response: str


In [ ]:
def find_sentiment(state: ReviewState):

    prompt = f"What is the sentiment of the following review -> \n {state['review']}"

    sentiment = model_with_structured_output.invoke(prompt).sentiment

    return {"sentiment": sentiment}

In [ ]:
def check_sentiment(state: ReviewState) -> Literal["generate_positive_response", "run_diagnosis"]:
    if state['sentiment'] == "positive":
        return "generate_positive_response"
    else:
        return "run_diagnosis"

In [ ]:
def generate_positive_response(state: ReviewState):
    
    prompt = f"Generate a positive response based on the following review -> {state['review']}"
    
    response = model.invoke(prompt).content
    
    return {"response": response}

In [ ]:
class DiagnosisSchema(BaseModel):
    issue_type: str
    tone: str
    urgency: str

In [ ]:
diagnosis_schema_structured_model = model.with_structured_output(DiagnosisSchema)

In [ ]:
def run_diagnosis(state: ReviewState):
    prompt = f"Based on the following review , generate a diagnosis based on the sentiment of the review -> {state['review']}"
    
    response = diagnosis_schema_structured_model.invoke(prompt)
    
    return {
        "diagnosis" : response.model_dump()
        }
    

In [ ]:
def generate_negative_response(state: ReviewState):
    
    prompt = f"Generate a negative response based on the following review and diagnosis: \n Review -> {state['review']} \n Diagnosis -> {state['diagnosis']}"
    
    response = model.invoke(prompt).content
    
    return {"response": response}

In [ ]:
graph = StateGraph(ReviewState)

graph.add_node("find_sentiment", find_sentiment)
graph.add_node("generate_negative_response", generate_negative_response)
graph.add_node("run_diagnosis", run_diagnosis)
graph.add_node("generate_positive_response", generate_positive_response)

graph.add_edge(START, "find_sentiment")
graph.add_conditional_edges("find_sentiment", check_sentiment)
graph.add_edge("generate_positive_response", END)
graph.add_edge("run_diagnosis", "generate_negative_response")
graph.add_edge("generate_negative_response", END)

workflow =  graph.compile()
workflow

In [ ]:
# initial_state = {
#     "review": "The speaker of this device is so bad"
# }

# response = workflow.invoke(initial_state)
# response

In [ ]:
# test_reviews = [
    # Positive
    # "Absolutely love these headphones! The noise cancellation is incredible and the battery easily lasts two full days.",
    # "Setup took less than five minutes and the app is super intuitive. Best smart home purchase I've made this year.",
    # "Customer support was fantastic - they replaced my faulty charger within 48 hours, no questions asked.",
    # "The camera quality on this phone is stunning, especially in low light. Worth every penny.",
    # "Lightweight, fast, and the keyboard feels great to type on. Perfect laptop for students.",

    # Negative - product defect
    # "The screen started flickering after just one week of use. Really disappointed with the build quality.",
    # "Battery drains from 100% to 20% in under three hours even when idle. This is unacceptable for a new device.",

    # Negative - software / bug
    # "The app crashes every time I try to upload a photo. I've reinstalled it three times and it still doesn't work.",
    # "After the latest update, I can't log in at all. It keeps saying 'invalid credentials' even after resetting my password.",

    # Negative - delivery / service
    # "Ordered two weeks ago and the package still hasn't arrived. Tracking hasn't updated in 8 days.",
    # "I was charged twice for the same order and support has ignored my emails for a week. I want my money back NOW.",

    # Negative - mild / calm tone
    # "The product is okay, but the instructions were confusing and a few screws were missing from the box.",

    # Mixed / edge cases
    # "Great sound quality, but the Bluetooth keeps disconnecting every few minutes which makes it almost unusable.",
    # "It's fine. Does what it says. Nothing special.",
    # "Oh great, another 'premium' charger that stopped working in two days. Fantastic job, really.",  # sarcasm
# ]

In [ ]:
pos_review = """
    "Absolutely love these headphones! The noise cancellation is incredible and the battery easily lasts two full days.
    Setup took less than five minutes and the app is super intuitive. Best smart home purchase I've made this year.
    Customer support was fantastic - they replaced my faulty charger within 48 hours, no questions asked.The camera quality on this phone is stunning, especially in low light. Worth every penny.Lightweight, fast, and the keyboard feels great to type on. Perfect laptop for students.
"""

In [ ]:
neg_review = """
The screen started flickering after just one week of use. Really disappointed with the build quality.Battery drains from 100% to 20% in under three hours even when idle. This is unacceptable for a new device.
"""

In [ ]:
# for review in test_reviews:
#     result = workflow.invoke({"review": review})
#     print("Review    :", result["review"])
#     print("Sentiment :", result["sentiment"])
#     if "diagnosis" in result:
#         print("Diagnosis :", result["diagnosis"])
#     print("Response  :", result.get("response"))
#     print("-" * 100)

initial_state = {
    "review": neg_review
}

result =  workflow.invoke(initial_state)
result